# 中证800 V71 后验驱动因子替换与增强实验

独立版 notebook，可上传到 JoinQuant 研究环境运行。

实验目标：不改 V46 的训练方式，只根据 V69 后验诊断调整因子池，验证当前问题到底来自因子噪声、进攻型高波动暴露，还是组合层 top tail 敏感。

固定训练框架：`full/direct/fixed120/legacy_rebalance`。本 notebook 会比较原版 full、删除嫌疑因子、短周期风险替换、质量现金流增强、估值质量现金流增强、估值质量现金流 + 中长趋势。


## 0. 导入与进度条


In [ ]:
import os
import gc
import builtins as _bi
import warnings
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 260)
pd.set_option("display.width", 260)

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


def progress_iter(iterable, total=None, desc="progress", leave=True):
    if tqdm is not None:
        return tqdm(iterable, total=total, desc=desc, leave=leave)
    def _gen():
        every = _bi.max(1, int((total or 100) / 20))
        for i, item in enumerate(iterable, 1):
            if i == 1 or i % every == 0 or (total is not None and i == total):
                print("%s %s%s" % (desc, i, "/%s" % total if total else ""))
            yield item
    return _gen()


def display_df(df, n=30):
    try:
        display(df.head(n))
    except Exception:
        print(df.head(n).to_string(index=False))


## 1. 配置


In [ ]:
PROJECT_DIR = Path.cwd()
OUT_DIR = PROJECT_DIR / "csi800_ml_v71_factor_replacement_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_CANDIDATES = [
    Path("train_csi800_factor_v40_data_enhancement_20190101_20260531.csv"),
    Path("train_csi800_factor_v40_data_enhancement.csv"),
    Path("data/train_csi800_factor_v40_data_enhancement_20190101_20260531.csv"),
    Path("data/train_csi800_factor_v40_data_enhancement.csv"),
    PROJECT_DIR / "train_csi800_factor_v40_data_enhancement_20190101_20260531.csv",
    PROJECT_DIR / "train_csi800_factor_v40_data_enhancement.csv",
    PROJECT_DIR / "data" / "train_csi800_factor_v40_data_enhancement_20190101_20260531.csv",
    PROJECT_DIR / "data" / "train_csi800_factor_v40_data_enhancement.csv",
]
DATA_PATH_OVERRIDE = None

TARGET_COL = "alpha_1m"
STOCK_COL = "stock"
DATE_COL = "rebalance_date"
FACTOR_DATE_COL = "feature_date"
INDUSTRY_COL = "industry_bucket"

FIXED_ITER = 120
SEED = 42
CORR_THRESHOLD = 0.70
LABEL_BOUNDARY_MODE = "legacy_rebalance"

# JQ 上传运行时建议保持 True。如果新增因子已经在 CSV 里，可以设为 False 节省时间。
ENABLE_JQFACTOR_FETCH = True
JQFACTOR_CACHE_FILE = OUT_DIR / "v71_jqfactor_extra_cache.csv"
MAX_FACTORS_PER_CALL = 20
MAX_SECURITIES_PER_CALL = 900

SMOKE_TEST = False
SMOKE_MAX_VARIANTS = 2
SMOKE_MAX_MODELS = 1

# JQ research kernels can OOM if every scored row keeps all factor columns.
# Keep this True unless you explicitly need a full diagnostic panel.
LOW_MEMORY_MODE = True
SAVE_SCORE_PANEL = True

# If JQ kernel is still memory-constrained, run variants in batches, e.g.
# RUN_VARIANTS = ["full_v46", "replace_short_risk", "vqcf_midtrend"]
RUN_VARIANTS = None

DIAG_MODEL_SPECS = [
    {"model_tag": "exp_2021_12", "train_start": "2019-01-01", "train_end": "2021-12-31", "test_start": "2022-01-01", "test_end": "2023-12-31", "phase": "early_36m"},
    {"model_tag": "exp_2022_12", "train_start": "2019-01-01", "train_end": "2022-12-31", "test_start": "2023-01-01", "test_end": "2024-12-31", "phase": "transition_48m"},
    {"model_tag": "exp_2023_12", "train_start": "2019-01-01", "train_end": "2023-12-31", "test_start": "2024-01-01", "test_end": "2025-12-31", "phase": "robust_ge60m"},
    {"model_tag": "exp_2024_12", "train_start": "2019-01-01", "train_end": "2024-12-31", "test_start": "2025-01-01", "test_end": "2026-04-30", "phase": "robust_ge60m"},
    {"model_tag": "exp_2025_12", "train_start": "2019-01-01", "train_end": "2025-12-31", "test_start": "2026-01-01", "test_end": "2026-06-30", "phase": "robust_ge60m"},
]

if SMOKE_TEST:
    DIAG_MODEL_SPECS = DIAG_MODEL_SPECS[:SMOKE_MAX_MODELS]

PORTFOLIO_SPECS = [
    {"portfolio_rule": "top8_cap3_2", "stock_num": 8, "board_caps": {"chinext": 3, "star": 2}},
    {"portfolio_rule": "top15_cap6_4", "stock_num": 15, "board_caps": {"chinext": 6, "star": 4}},
]

MANUAL_FAILURE_MONTHS = [
    "2022-04-01", "2022-08-01", "2023-09-01", "2024-01-02", "2024-03-01",
    "2025-03-03", "2025-05-06", "2025-11-03", "2026-03-02",
]

print("OUT_DIR:", OUT_DIR)
print("fixed V46 training:", LABEL_BOUNDARY_MODE, "full/direct/fixed", FIXED_ITER)
print("ENABLE_JQFACTOR_FETCH:", ENABLE_JQFACTOR_FETCH)


## 2. V46 原始因子、嫌疑因子与新增候选因子


In [ ]:
BASE_FACTOR_COLS = [
    "cash_flow_to_price_ratio", "book_to_price_ratio", "earnings_yield", "sales_to_price_ratio",
    "cash_earnings_to_price_ratio", "earnings_to_price_ratio", "roe_ttm", "roa_ttm",
    "gross_profit_ttm", "operating_profit_to_total_profit", "net_operate_cash_flow_to_total_liability",
    "net_operating_cash_flow_coverage", "adjusted_profit_to_total_profit", "ACCA", "growth",
    "net_working_capital", "operating_profit_per_share", "net_operate_cash_flow_per_share",
    "total_operating_revenue_per_share", "super_quick_ratio", "MLEV", "debt_to_equity_ratio",
    "debt_to_tangible_equity_ratio", "momentum", "Rank1M", "sharpe_ratio_60", "Variance20",
    "liquidity", "beta", "ATR6", "MFI14", "DAVOL10", "VOL10", "VMACD", "VOSC",
    "Skewness20", "Kurtosis20",
]

HYBRID_LIGHT_EXTRA_COLS = [
    "liq_money_ratio_20_60", "liq_paused_count_20", "px_close_to_ma60", "px_drawdown_60",
    "ts_cash_flow_to_price_ratio_rank_mean_3m", "ts_Rank1M_rank_chg_1m",
]

FULL_V46_COLS = BASE_FACTOR_COLS + HYBRID_LIGHT_EXTRA_COLS

HOT_RISK_DROP = ["Variance20", "beta", "ATR6", "VOL10", "DAVOL10", "VOSC"]
SUSPECT_TECH_DROP = ["px_close_to_ma60", "liquidity", "sharpe_ratio_60", "MFI14", "Skewness20", "Kurtosis20"]

RISK_REPLACEMENT_FACTORS = [
    "Variance60", "Variance120", "sharpe_ratio_120", "turnover_volatility",
    "residual_volatility", "daily_standard_deviation", "cumulative_range", "historical_sigma",
]

QUALITY_CASHFLOW_FACTORS = [
    "cfo_to_ev", "net_operate_cash_flow_to_operate_income", "goods_service_cash_to_operating_revenue_ttm",
    "cash_rate_of_sales", "net_operate_cash_flow_to_asset", "net_operate_cashflow_growth_rate",
    "roic_ttm", "rnoa_ttm", "profit_margin_ttm", "asset_turnover_ttm", "roe_ttm_8y", "roa_ttm_8y",
    "margin_stability", "DEGM_8y", "maximum_margin", "gross_income_ratio", "operating_profit_ratio",
    "total_profit_to_cost_ratio",
]

VALUE_EXTRA_FACTORS = [
    "PEG", "predicted_earnings_to_price_ratio",
]

MID_TREND_FACTORS = [
    "MAC60", "MAC120", "EMAC120", "Price3M", "Price1Y", "ROC60", "ROC120", "fifty_two_week_close_rank",
]

VALUATION_CORE = [
    "cash_flow_to_price_ratio", "book_to_price_ratio", "earnings_yield", "sales_to_price_ratio",
    "cash_earnings_to_price_ratio", "earnings_to_price_ratio",
]
QUALITY_CORE = [
    "roe_ttm", "roa_ttm", "gross_profit_ttm", "operating_profit_to_total_profit", "adjusted_profit_to_total_profit", "ACCA",
]
CASHFLOW_CORE = [
    "net_operate_cash_flow_to_total_liability", "net_operating_cash_flow_coverage",
    "net_operate_cash_flow_per_share", "net_working_capital", "super_quick_ratio",
]
TEMPORAL_CORE = ["ts_cash_flow_to_price_ratio_rank_mean_3m", "ts_Rank1M_rank_chg_1m"]
MID_TREND_CORE = ["Rank1M", "momentum"]

FACTOR_GROUPS = {
    "valuation": VALUATION_CORE + VALUE_EXTRA_FACTORS,
    "profit_quality": QUALITY_CORE + ["roic_ttm", "rnoa_ttm", "profit_margin_ttm", "asset_turnover_ttm", "roe_ttm_8y", "roa_ttm_8y", "margin_stability", "DEGM_8y", "maximum_margin", "gross_income_ratio", "operating_profit_ratio", "total_profit_to_cost_ratio"],
    "cashflow_balance": CASHFLOW_CORE + ["cfo_to_ev", "net_operate_cash_flow_to_operate_income", "goods_service_cash_to_operating_revenue_ttm", "cash_rate_of_sales", "net_operate_cash_flow_to_asset", "net_operate_cashflow_growth_rate"],
    "growth_income": ["growth", "operating_profit_per_share", "total_operating_revenue_per_share"],
    "leverage_risk": ["MLEV", "debt_to_equity_ratio", "debt_to_tangible_equity_ratio", "beta", "Variance20", "ATR6", "Variance60", "Variance120", "sharpe_ratio_120", "turnover_volatility", "residual_volatility", "daily_standard_deviation", "cumulative_range", "historical_sigma"],
    "momentum_technical": ["momentum", "Rank1M", "sharpe_ratio_60", "MFI14", "DAVOL10", "VOL10", "VMACD", "VOSC", "Skewness20", "Kurtosis20"] + MID_TREND_FACTORS,
    "liquidity_price_path": ["liquidity", "liq_money_ratio_20_60", "liq_paused_count_20", "px_close_to_ma60", "px_drawdown_60"],
    "temporal": TEMPORAL_CORE,
}

BASE_PARAMS_FF10 = {
    "objective": "regression",
    "metric": "l2",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_data_in_leaf": 200,
    "feature_fraction": 1.0,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l1": 0.1,
    "lambda_l2": 0.3,
    "verbose": -1,
}


def unique_keep_order(cols):
    seen = set()
    out = []
    for col in cols:
        if col not in seen:
            out.append(col)
            seen.add(col)
    return out


def minus_cols(cols, drops):
    drop_set = set(drops)
    return [c for c in cols if c not in drop_set]


FACTOR_VARIANTS = [
    {
        "variant": "full_v46",
        "description": "current V46 full baseline",
        "candidate_cols": unique_keep_order(FULL_V46_COLS),
    },
    {
        "variant": "drop_hot_risk",
        "description": "drop high suspicion short risk/volume factors: Variance20 beta ATR6 VOL10 DAVOL10 VOSC",
        "candidate_cols": unique_keep_order(minus_cols(FULL_V46_COLS, HOT_RISK_DROP)),
    },
    {
        "variant": "drop_suspect_all",
        "description": "drop hot risk plus unstable technical/path factors",
        "candidate_cols": unique_keep_order(minus_cols(FULL_V46_COLS, HOT_RISK_DROP + SUSPECT_TECH_DROP)),
    },
    {
        "variant": "replace_short_risk",
        "description": "drop hot risk and add medium/long risk replacement factors",
        "candidate_cols": unique_keep_order(minus_cols(FULL_V46_COLS, HOT_RISK_DROP) + RISK_REPLACEMENT_FACTORS),
    },
    {
        "variant": "quality_cashflow_plus",
        "description": "full V46 plus quality/cashflow enhanced factors",
        "candidate_cols": unique_keep_order(FULL_V46_COLS + QUALITY_CASHFLOW_FACTORS),
    },
    {
        "variant": "value_quality_cashflow",
        "description": "valuation + quality + cashflow + temporal only, no broad technical/risk bucket",
        "candidate_cols": unique_keep_order(VALUATION_CORE + QUALITY_CORE + CASHFLOW_CORE + TEMPORAL_CORE + VALUE_EXTRA_FACTORS + QUALITY_CASHFLOW_FACTORS),
    },
    {
        "variant": "vqcf_midtrend",
        "description": "value/quality/cashflow plus limited medium/long trend",
        "candidate_cols": unique_keep_order(VALUATION_CORE + QUALITY_CORE + CASHFLOW_CORE + TEMPORAL_CORE + VALUE_EXTRA_FACTORS + QUALITY_CASHFLOW_FACTORS + MID_TREND_CORE + MID_TREND_FACTORS),
    },
]

if SMOKE_TEST:
    FACTOR_VARIANTS = FACTOR_VARIANTS[:SMOKE_MAX_VARIANTS]

if RUN_VARIANTS is not None:
    run_variant_set = set(RUN_VARIANTS)
    FACTOR_VARIANTS = [v for v in FACTOR_VARIANTS if v["variant"] in run_variant_set]

all_candidate_factor_list = []
for _variant_cfg in FACTOR_VARIANTS:
    all_candidate_factor_list.extend(_variant_cfg["candidate_cols"])
ALL_CANDIDATE_FACTORS = unique_keep_order(all_candidate_factor_list)
EXTRA_JQFACTORS = unique_keep_order([c for c in ALL_CANDIDATE_FACTORS if c not in FULL_V46_COLS])

print("variants:", [v["variant"] for v in FACTOR_VARIANTS])
print("extra jqfactor candidates:", len(EXTRA_JQFACTORS), EXTRA_JQFACTORS)


## 3. 通用函数


In [ ]:
def resolve_data_path():
    if DATA_PATH_OVERRIDE:
        p = Path(DATA_PATH_OVERRIDE)
        if p.exists():
            return p
        raise IOError("DATA_PATH_OVERRIDE not found: %s" % p)
    candidates = [Path(x) for x in DATA_CANDIDATES]
    for p in candidates:
        if p.exists():
            return p
    searched = [str(p.resolve()) for p in candidates]
    raise IOError("training data csv not found. Put train_csi800_factor_v40_data_enhancement*.csv in one of: %s" % searched)


def safe_to_datetime(df, cols):
    out = df.copy()
    for col in cols:
        if col in out.columns:
            out[col] = pd.to_datetime(out[col], errors="coerce").dt.normalize()
    return out


def safe_rank_ic(a, b):
    s = pd.DataFrame({"a": np.asarray(a, dtype=float), "b": np.asarray(b, dtype=float)})
    s = s.replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < 3 or s["a"].nunique() < 2 or s["b"].nunique() < 2:
        return np.nan
    return s["a"].rank(pct=True).corr(s["b"].rank(pct=True))


def calc_nav(ret_series):
    s = pd.Series(ret_series).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    if len(s) == 0:
        return pd.Series(dtype=float)
    return (1.0 + s).cumprod()


def calc_max_drawdown(ret_series):
    nav = calc_nav(ret_series)
    if len(nav) == 0:
        return np.nan
    return float((nav / nav.cummax() - 1.0).min())


def summarize_returns(ret_series):
    s = pd.Series(ret_series).replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) == 0:
        return {"months": 0, "cum_ret": np.nan, "mean_ret": np.nan, "win_rate": np.nan, "max_drawdown": np.nan, "worst_month": np.nan}
    return {
        "months": int(len(s)),
        "cum_ret": float((1.0 + s).prod() - 1.0),
        "mean_ret": float(s.mean()),
        "win_rate": float((s > 0).mean()),
        "max_drawdown": calc_max_drawdown(s),
        "worst_month": float(s.min()),
    }


def feature_group_of(feature):
    for g, cols in FACTOR_GROUPS.items():
        if feature in cols:
            return g
    return "other"


def build_corr_components(train_df, feature_cols, threshold):
    from collections import defaultdict
    corr = train_df[feature_cols].corr()
    graph = defaultdict(list)
    for i in range(len(feature_cols)):
        for j in range(i + 1, len(feature_cols)):
            v = corr.iloc[i, j]
            if not pd.isnull(v) and abs(v) > threshold:
                graph[feature_cols[i]].append(feature_cols[j])
                graph[feature_cols[j]].append(feature_cols[i])
    for col in feature_cols:
        graph[col]
    visited = set()
    comps = []

    def dfs(x, comp):
        visited.add(x)
        comp.append(x)
        for y in graph[x]:
            if y not in visited:
                dfs(y, comp)

    for col in feature_cols:
        if col not in visited:
            comp = []
            dfs(col, comp)
            comps.append(comp)
    return comps


def select_features_train_only(train_df, candidate_cols):
    cols = unique_keep_order([c for c in candidate_cols if c in train_df.columns])
    if len(cols) == 0:
        raise ValueError("no candidate feature exists in train data")
    missing = train_df[cols].isnull().sum().to_dict()
    keep = []
    remove = []
    for comp in build_corr_components(train_df, cols, CORR_THRESHOLD):
        if len(comp) == 1:
            keep.append(comp[0])
        else:
            comp = _bi.sorted(comp, key=lambda x: (missing[x], x))
            keep.append(comp[0])
            remove.extend(comp[1:])
    return keep, remove


def prepare_xy(df, feature_cols, target_col, fill_values=None):
    d = df.dropna(subset=[target_col]).copy()
    X = d.reindex(columns=feature_cols).replace([np.inf, -np.inf], np.nan).copy()
    y = d[target_col].astype(float).copy()
    if fill_values is None:
        fill_values = X.median().replace([np.inf, -np.inf], np.nan).fillna(0)
    X = X.fillna(fill_values).fillna(0)
    return X, y, fill_values, d.index


def make_train_df(df_all, spec):
    start = pd.Timestamp(spec["train_start"])
    end = pd.Timestamp(spec["train_end"])
    mask = (df_all[DATE_COL] >= start) & (df_all[DATE_COL] <= end)
    if LABEL_BOUNDARY_MODE == "label_end_safe":
        mask = mask & (df_all["next_date"] <= end)
    elif LABEL_BOUNDARY_MODE != "legacy_rebalance":
        raise ValueError("unknown LABEL_BOUNDARY_MODE: " + str(LABEL_BOUNDARY_MODE))
    return df_all[mask].copy()


def make_test_df(df_all, spec):
    start = pd.Timestamp(spec["test_start"])
    end = pd.Timestamp(spec["test_end"])
    return df_all[(df_all[DATE_COL] >= start) & (df_all[DATE_COL] <= end)].copy()


def train_direct_lgb(train_df, feature_cols):
    params = dict(BASE_PARAMS_FF10)
    params["seed"] = SEED
    X_train, y_train, fill_values, _ = prepare_xy(train_df, feature_cols, TARGET_COL)
    if len(X_train) == 0:
        raise ValueError("empty training matrix")
    model = lgb.train(params, lgb.Dataset(X_train, label=y_train), num_boost_round=_bi.max(1, int(FIXED_ITER)))
    pred = np.asarray(model.predict(X_train[feature_cols], num_iteration=FIXED_ITER)).reshape(-1)
    return {"model": model, "fill_values": fill_values, "train_rows": int(len(X_train)), "train_rank_ic": safe_rank_ic(y_train, pred)}


def score_with_model(df, model, feature_cols, fill_values):
    X = df.reindex(columns=feature_cols).replace([np.inf, -np.inf], np.nan).copy()
    X = X.fillna(fill_values).fillna(0)
    return np.asarray(model.predict(X[feature_cols], num_iteration=FIXED_ITER)).reshape(-1)


def get_stock_board(stock):
    code = str(stock).split(".")[0]
    if code.startswith(("300", "301")):
        return "chinext"
    if code.startswith(("688", "689")):
        return "star"
    return "main"


def board_cap_allows(selected, stock, board_caps):
    board = get_stock_board(stock)
    if board not in board_caps:
        return True
    current = sum(1 for s in selected if get_stock_board(s) == board)
    return current < int(board_caps[board])


def build_board_capped_targets(sorted_stocks, target_num, board_caps):
    selected = []
    for stock in sorted_stocks:
        if stock in selected:
            continue
        if board_cap_allows(selected, stock, board_caps):
            selected.append(stock)
            if len(selected) >= int(target_num):
                return selected
    for stock in sorted_stocks:
        if stock not in selected:
            selected.append(stock)
            if len(selected) >= int(target_num):
                break
    return selected


def load_dataset(path):
    df = pd.read_csv(path)
    df = safe_to_datetime(df, [DATE_COL, FACTOR_DATE_COL, "next_date"])
    if STOCK_COL not in df.columns:
        for alt in ["code", "security", "order_book_id"]:
            if alt in df.columns:
                df = df.rename(columns={alt: STOCK_COL})
                break
    if TARGET_COL not in df.columns:
        if "raw_return_1m" in df.columns and "benchmark_csi800_1m" in df.columns:
            df[TARGET_COL] = df["raw_return_1m"] - df["benchmark_csi800_1m"]
        else:
            raise ValueError("target column not found: " + TARGET_COL)
    if FACTOR_DATE_COL not in df.columns:
        df[FACTOR_DATE_COL] = df[DATE_COL]
    if INDUSTRY_COL not in df.columns:
        df[INDUSTRY_COL] = "UNKNOWN"
    need = [STOCK_COL, DATE_COL, TARGET_COL, INDUSTRY_COL, FACTOR_DATE_COL, "next_date"]
    missing = [c for c in need if c not in df.columns]
    if missing:
        raise ValueError("dataset missing columns: " + ",".join(missing))
    df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
    df[STOCK_COL] = df[STOCK_COL].astype(str)
    df = df.dropna(subset=[STOCK_COL, DATE_COL, FACTOR_DATE_COL, TARGET_COL]).copy()
    df["factor_query_date"] = pd.to_datetime(df[FACTOR_DATE_COL], errors="coerce").dt.normalize()
    return df


## 4. 可选：从 jqfactor 补充新增因子


In [ ]:
def chunk_list(items, chunk_size):
    chunk = []
    for x in items:
        chunk.append(x)
        if len(chunk) >= chunk_size:
            yield chunk
            chunk = []
    if chunk:
        yield chunk


def fetch_jqfactor_for_panel(df, factors):
    if len(factors) == 0:
        return pd.DataFrame()
    try:
        from jqfactor import get_factor_values
    except Exception as e:
        print("jqfactor unavailable, skip extra factor fetch:", repr(e))
        return pd.DataFrame()

    query_dates = _bi.sorted(pd.to_datetime(df["factor_query_date"].dropna().unique()))
    rows = []
    status_rows = []
    date_iter = progress_iter(query_dates, total=len(query_dates), desc="fetch jqfactor by date")
    for dt in date_iter:
        dstr = pd.Timestamp(dt).strftime("%Y-%m-%d")
        sec = df[df["factor_query_date"] == dt][STOCK_COL].astype(str).drop_duplicates().tolist()
        if len(sec) == 0:
            continue
        for sec_batch in chunk_list(sec, MAX_SECURITIES_PER_CALL):
            for fac_batch in chunk_list(factors, MAX_FACTORS_PER_CALL):
                try:
                    dic = get_factor_values(sec_batch, fac_batch, end_date=dstr, count=1)
                    tmp = pd.DataFrame({STOCK_COL: sec_batch})
                    tmp["factor_query_date"] = pd.Timestamp(dt).normalize()
                    for fac in fac_batch:
                        if fac in dic:
                            mat = dic[fac]
                            if mat is not None and len(mat) > 0:
                                s = mat.iloc[-1]
                                tmp[fac] = tmp[STOCK_COL].map(s.to_dict())
                            else:
                                tmp[fac] = np.nan
                        else:
                            tmp[fac] = np.nan
                    rows.append(tmp)
                    status_rows.append({"factor_query_date": dstr, "securities": len(sec_batch), "factors": ",".join(fac_batch), "status": "ok", "error": ""})
                except Exception as e:
                    status_rows.append({"factor_query_date": dstr, "securities": len(sec_batch), "factors": ",".join(fac_batch), "status": "batch_error_retry_single", "error": repr(e)})
                    print("fetch batch error, retry one by one", dstr, fac_batch, repr(e))
                    for fac in fac_batch:
                        try:
                            dic1 = get_factor_values(sec_batch, [fac], end_date=dstr, count=1)
                            tmp = pd.DataFrame({STOCK_COL: sec_batch})
                            tmp["factor_query_date"] = pd.Timestamp(dt).normalize()
                            if fac in dic1 and dic1[fac] is not None and len(dic1[fac]) > 0:
                                s = dic1[fac].iloc[-1]
                                tmp[fac] = tmp[STOCK_COL].map(s.to_dict())
                            else:
                                tmp[fac] = np.nan
                            rows.append(tmp)
                            status_rows.append({"factor_query_date": dstr, "securities": len(sec_batch), "factors": fac, "status": "single_ok", "error": ""})
                        except Exception as e1:
                            status_rows.append({"factor_query_date": dstr, "securities": len(sec_batch), "factors": fac, "status": "single_error", "error": repr(e1)})
                            print("fetch single error", dstr, fac, repr(e1))
    status_df = pd.DataFrame(status_rows)
    status_df.to_csv(OUT_DIR / "v71_jqfactor_fetch_status.csv", index=False)
    if len(rows) == 0:
        return pd.DataFrame()
    out = pd.concat(rows, ignore_index=True)
    out = out.groupby([STOCK_COL, "factor_query_date"], as_index=False).first()
    return out


def maybe_add_extra_factors(df):
    need_extra = [c for c in EXTRA_JQFACTORS if c not in df.columns]
    if len(need_extra) == 0:
        print("all extra factors already in dataset")
        return df
    print("missing extra factors:", len(need_extra), need_extra)
    cache_df = pd.DataFrame()
    if JQFACTOR_CACHE_FILE.exists():
        try:
            cache_df = pd.read_csv(JQFACTOR_CACHE_FILE)
            cache_df = safe_to_datetime(cache_df, ["factor_query_date"])
            print("loaded jqfactor cache:", cache_df.shape)
        except Exception as e:
            print("cache read failed, refetch:", repr(e))
            cache_df = pd.DataFrame()
    if ENABLE_JQFACTOR_FETCH:
        have_cols = list(cache_df.columns) if len(cache_df) else []
        still_need = [c for c in need_extra if c not in have_cols]
        if len(still_need) > 0 or len(cache_df) == 0:
            fetched = fetch_jqfactor_for_panel(df, need_extra)
            if len(fetched):
                cache_df = fetched
                cache_df.to_csv(JQFACTOR_CACHE_FILE, index=False)
                print("saved jqfactor cache:", JQFACTOR_CACHE_FILE, cache_df.shape)
    if len(cache_df) == 0:
        print("no jqfactor extras merged; variants will use available columns only")
        return df
    merge_cols = [STOCK_COL, "factor_query_date"] + [c for c in need_extra if c in cache_df.columns]
    cache_df = cache_df[merge_cols].copy()
    out = df.merge(cache_df, on=[STOCK_COL, "factor_query_date"], how="left")
    return out


## 5. 读取数据、补因子、生成 variant manifest


In [ ]:
DATA_PATH = resolve_data_path()
df_all = load_dataset(DATA_PATH)
print("DATA_PATH:", DATA_PATH)
print("loaded:", df_all.shape)
print("rebalance_date:", df_all[DATE_COL].min(), "->", df_all[DATE_COL].max())
print("factor_query_date:", df_all["factor_query_date"].min(), "->", df_all["factor_query_date"].max())

df_all = maybe_add_extra_factors(df_all)
print("after extra factors:", df_all.shape)

variant_rows = []
usable_variants = []
for v in FACTOR_VARIANTS:
    cols = unique_keep_order([c for c in v["candidate_cols"] if c in df_all.columns])
    missing = [c for c in v["candidate_cols"] if c not in df_all.columns]
    row = {
        "variant": v["variant"],
        "description": v["description"],
        "candidate_count": len(v["candidate_cols"]),
        "available_count": len(cols),
        "missing_count": len(missing),
        "available_cols": ",".join(cols),
        "missing_cols": ",".join(missing),
    }
    variant_rows.append(row)
    if len(cols) >= 10:
        nv = dict(v)
        nv["available_cols"] = cols
        nv["missing_cols"] = missing
        usable_variants.append(nv)
    else:
        print("skip variant too few cols", v["variant"], len(cols), missing)

variant_manifest_df = pd.DataFrame(variant_rows)
variant_manifest_df.to_csv(OUT_DIR / "v71_variant_manifest.csv", index=False)
display_df(variant_manifest_df[["variant", "candidate_count", "available_count", "missing_count", "missing_cols"]], 20)

print("usable variants:", [v["variant"] for v in usable_variants])


## 6. 训练所有因子变体并打分


In [ ]:
model_meta_rows = []
feature_importance_rows = []
group_importance_rows = []
score_panel_parts = []

total_jobs = len(usable_variants) * len(DIAG_MODEL_SPECS)
job_iter = []
for v in usable_variants:
    for spec in DIAG_MODEL_SPECS:
        job_iter.append((v, spec))

for v, spec in progress_iter(job_iter, total=len(job_iter), desc="train V71 variant models"):
    variant = v["variant"]
    train_df = make_train_df(df_all, spec)
    test_df = make_test_df(df_all, spec)
    if train_df.empty or test_df.empty:
        print("skip empty", variant, spec["model_tag"], train_df.shape, test_df.shape)
        continue
    feature_cols, removed_cols = select_features_train_only(train_df, v["available_cols"])
    if len(feature_cols) < 10:
        print("skip too few selected features", variant, spec["model_tag"], len(feature_cols))
        continue
    trained = train_direct_lgb(train_df, feature_cols)
    pred_score = score_with_model(test_df, trained["model"], feature_cols, trained["fill_values"])
    keep_cols = [STOCK_COL, DATE_COL, TARGET_COL]
    for _extra_col in [INDUSTRY_COL, "next_date", FACTOR_DATE_COL, "factor_query_date"]:
        if _extra_col in test_df.columns and _extra_col not in keep_cols:
            keep_cols.append(_extra_col)
    for _extra_col in [
        "beta", "Variance20", "ATR6", "VOL10", "DAVOL10", "VOSC", "liquidity", "momentum", "Rank1M",
        "book_to_price_ratio", "earnings_yield", "cash_flow_to_price_ratio", "px_close_to_ma60", "px_drawdown_60",
    ]:
        if _extra_col in test_df.columns and _extra_col not in keep_cols:
            keep_cols.append(_extra_col)
    score_df = test_df[keep_cols].copy()
    score_df["score"] = pred_score
    score_df["variant"] = variant
    score_df["model_tag"] = spec["model_tag"]
    score_df["phase"] = spec.get("phase", "")
    score_df["train_start"] = pd.Timestamp(spec["train_start"])
    score_df["train_end"] = pd.Timestamp(spec["train_end"])
    score_panel_parts.append(score_df)

    model_meta_rows.append({
        "variant": variant,
        "model_tag": spec["model_tag"],
        "phase": spec.get("phase", ""),
        "train_start": spec["train_start"],
        "train_end": spec["train_end"],
        "test_start": spec["test_start"],
        "test_end": spec["test_end"],
        "train_months": int(train_df[DATE_COL].nunique()),
        "train_rows": int(len(train_df)),
        "test_months": int(test_df[DATE_COL].nunique()),
        "test_rows": int(len(test_df)),
        "available_feature_count": int(len(v["available_cols"])),
        "selected_feature_count": int(len(feature_cols)),
        "removed_feature_count": int(len(removed_cols)),
        "train_rank_ic": trained["train_rank_ic"],
        "feature_cols": ",".join(feature_cols),
        "removed_features": ",".join(removed_cols),
    })

    gains = trained["model"].feature_importance(importance_type="gain")
    splits = trained["model"].feature_importance(importance_type="split")
    imp_df = pd.DataFrame({"feature": feature_cols, "importance_gain": gains, "importance_split": splits})
    total_gain = float(imp_df["importance_gain"].sum()) if len(imp_df) else 0.0
    total_split = float(imp_df["importance_split"].sum()) if len(imp_df) else 0.0
    imp_df["importance_gain_pct"] = imp_df["importance_gain"] / total_gain if total_gain > 0 else np.nan
    imp_df["importance_split_pct"] = imp_df["importance_split"] / total_split if total_split > 0 else np.nan
    imp_df["feature_group"] = imp_df["feature"].map(feature_group_of)
    imp_df["variant"] = variant
    imp_df["model_tag"] = spec["model_tag"]
    imp_df["train_end"] = spec["train_end"]
    feature_importance_rows.extend(imp_df.to_dict("records"))

    grp_gain = imp_df.groupby("feature_group")["importance_gain"].sum().reset_index().rename(columns={"importance_gain": "importance_gain"})
    grp_split = imp_df.groupby("feature_group")["importance_split"].sum().reset_index().rename(columns={"importance_split": "importance_split"})
    grp_count = imp_df.groupby("feature_group")["feature"].size().reset_index().rename(columns={"feature": "feature_count"})
    grp = grp_gain.merge(grp_split, on="feature_group", how="outer").merge(grp_count, on="feature_group", how="outer")
    grp["importance_gain_pct"] = grp["importance_gain"] / grp["importance_gain"].sum() if grp["importance_gain"].sum() > 0 else np.nan
    grp["importance_split_pct"] = grp["importance_split"] / grp["importance_split"].sum() if grp["importance_split"].sum() > 0 else np.nan
    grp["variant"] = variant
    grp["model_tag"] = spec["model_tag"]
    grp["train_end"] = spec["train_end"]
    group_importance_rows.extend(grp.to_dict("records"))
    del trained, train_df, test_df, score_df
    gc.collect()

model_meta_df = pd.DataFrame(model_meta_rows)
feature_importance_df = pd.DataFrame(feature_importance_rows)
group_importance_df = pd.DataFrame(group_importance_rows)
score_panel_df = pd.concat(score_panel_parts, ignore_index=True) if score_panel_parts else pd.DataFrame()

model_meta_df.to_csv(OUT_DIR / "v71_model_meta.csv", index=False)
feature_importance_df.to_csv(OUT_DIR / "v71_feature_importance.csv", index=False)
group_importance_df.to_csv(OUT_DIR / "v71_feature_group_importance.csv", index=False)
if SAVE_SCORE_PANEL:
    score_panel_df.to_csv(OUT_DIR / "v71_score_panel.csv", index=False)

print("score_panel:", score_panel_df.shape)
display_df(model_meta_df[["variant", "model_tag", "phase", "train_months", "selected_feature_count", "train_rank_ic"]], 40)


## 7. 组合收益、bucket 延展性与失败月


In [ ]:
def summarize_group(df, keys, ret_col="mean_alpha"):
    rows = []
    if len(df) == 0:
        return pd.DataFrame()
    for name, gdf in df.groupby(keys):
        if not isinstance(name, tuple):
            name = (name,)
        row = {}
        for i, key in enumerate(keys):
            row[key] = name[i]
        stats = summarize_returns(gdf[ret_col])
        for k, val in stats.items():
            row[k] = val
        row["avg_monthly_alpha"] = float(pd.to_numeric(gdf[ret_col], errors="coerce").mean())
        row["monthly_win_rate"] = float((pd.to_numeric(gdf[ret_col], errors="coerce") > 0).mean())
        if "turnover" in gdf.columns:
            row["avg_turnover"] = float(pd.to_numeric(gdf["turnover"], errors="coerce").mean())
        if "rank_ic" in gdf.columns:
            row["avg_rank_ic"] = float(pd.to_numeric(gdf["rank_ic"], errors="coerce").mean())
        rows.append(row)
    return pd.DataFrame(rows)


portfolio_rows = []
bucket_rows = []
failure_rows = []
failure_dates = set(pd.to_datetime(MANUAL_FAILURE_MONTHS).normalize())

grouped_score = score_panel_df.groupby(["variant", "model_tag", DATE_COL]) if len(score_panel_df) else []
ngroups = score_panel_df.groupby(["variant", "model_tag", DATE_COL]).ngroups if len(score_panel_df) else 0
for (variant, model_tag, dt), gdf in progress_iter(grouped_score, total=ngroups, desc="portfolio and buckets"):
    m = gdf.dropna(subset=["score", TARGET_COL]).copy()
    if m.empty:
        continue
    m[STOCK_COL] = m[STOCK_COL].astype(str)
    ordered = m.sort_values("score", ascending=False).reset_index(drop=True)
    ordered_stocks = ordered[STOCK_COL].tolist()
    ret_map = dict(zip(m[STOCK_COL], pd.to_numeric(m[TARGET_COL], errors="coerce")))
    rank_ic = safe_rank_ic(m["score"], m[TARGET_COL])

    bucket_defs = [("top8_raw", 0, 8), ("rank09_20", 8, 20), ("rank21_50", 20, 50), ("rank51_100", 50, 100), ("bottom50", _bi.max(0, len(ordered) - 50), len(ordered))]
    for bname, a, b in bucket_defs:
        sub = ordered.iloc[a:_bi.min(b, len(ordered))].copy()
        if len(sub) == 0:
            continue
        bucket_rows.append({
            "variant": variant,
            "model_tag": model_tag,
            "rebalance_date": dt,
            "bucket": bname,
            "count": int(len(sub)),
            "mean_alpha": float(pd.to_numeric(sub[TARGET_COL], errors="coerce").mean()),
            "median_alpha": float(pd.to_numeric(sub[TARGET_COL], errors="coerce").median()),
            "rank_ic": rank_ic,
            "targets": ",".join(sub[STOCK_COL].astype(str).tolist()[:30]),
        })

    for pspec in PORTFOLIO_SPECS:
        targets = build_board_capped_targets(ordered_stocks, pspec["stock_num"], pspec["board_caps"])
        target_rets = [ret_map.get(s, np.nan) for s in targets]
        board_counts = {"main": 0, "chinext": 0, "star": 0}
        for s in targets:
            b = get_stock_board(s)
            board_counts[b] = board_counts.get(b, 0) + 1
        row = {
            "variant": variant,
            "model_tag": model_tag,
            "rebalance_date": dt,
            "portfolio_rule": pspec["portfolio_rule"],
            "stock_num": int(pspec["stock_num"]),
            "mean_alpha": float(np.nanmean(target_rets)) if len(target_rets) else np.nan,
            "median_alpha": float(np.nanmedian(target_rets)) if len(target_rets) else np.nan,
            "rank_ic": rank_ic,
            "board_main": int(board_counts.get("main", 0)),
            "board_chinext": int(board_counts.get("chinext", 0)),
            "board_star": int(board_counts.get("star", 0)),
            "targets": ",".join(targets),
        }
        portfolio_rows.append(row)
        if pd.Timestamp(dt).normalize() in failure_dates:
            frow = dict(row)
            frow["manual_failure_month"] = True
            failure_rows.append(frow)

portfolio_monthly_df = pd.DataFrame(portfolio_rows)
if len(portfolio_monthly_df):
    portfolio_monthly_df = portfolio_monthly_df.sort_values(["variant", "model_tag", "portfolio_rule", "rebalance_date"]).reset_index(drop=True)
    turnover_rows = []
    for (variant, model_tag, rule), gdf in portfolio_monthly_df.groupby(["variant", "model_tag", "portfolio_rule"]):
        prev = None
        for idx, row in gdf.sort_values("rebalance_date").iterrows():
            cur = set(str(row["targets"]).split(",")) if str(row["targets"]) else set()
            if prev is None or len(cur) == 0:
                turnover = np.nan
                overlap = np.nan
            else:
                overlap = len(cur & prev) / float(_bi.max(1, len(cur)))
                turnover = 1.0 - overlap
            turnover_rows.append({"idx": idx, "overlap_prev": overlap, "turnover": turnover})
            prev = cur
    turnover_df = pd.DataFrame(turnover_rows).set_index("idx") if turnover_rows else pd.DataFrame()
    if len(turnover_df):
        portfolio_monthly_df["overlap_prev"] = turnover_df["overlap_prev"]
        portfolio_monthly_df["turnover"] = turnover_df["turnover"]

bucket_monthly_df = pd.DataFrame(bucket_rows)
failure_month_df = pd.DataFrame(failure_rows)

portfolio_summary_df = summarize_group(portfolio_monthly_df, ["variant", "model_tag", "portfolio_rule", "stock_num"])
bucket_summary_df = summarize_group(bucket_monthly_df, ["variant", "model_tag", "bucket"])
robust_portfolio_df = portfolio_monthly_df[portfolio_monthly_df["model_tag"].isin(["exp_2023_12", "exp_2024_12", "exp_2025_12"])].copy() if len(portfolio_monthly_df) else pd.DataFrame()
robust_summary_df = summarize_group(robust_portfolio_df, ["variant", "portfolio_rule", "stock_num"])
robust_bucket_df = bucket_monthly_df[bucket_monthly_df["model_tag"].isin(["exp_2023_12", "exp_2024_12", "exp_2025_12"])].copy() if len(bucket_monthly_df) else pd.DataFrame()
robust_bucket_summary_df = summarize_group(robust_bucket_df, ["variant", "bucket"])
failure_summary_df = summarize_group(failure_month_df, ["variant", "portfolio_rule", "stock_num"])

portfolio_monthly_df.to_csv(OUT_DIR / "v71_portfolio_monthly.csv", index=False)
portfolio_summary_df.to_csv(OUT_DIR / "v71_portfolio_summary.csv", index=False)
robust_summary_df.to_csv(OUT_DIR / "v71_robust_phase_summary.csv", index=False)
bucket_monthly_df.to_csv(OUT_DIR / "v71_bucket_monthly.csv", index=False)
bucket_summary_df.to_csv(OUT_DIR / "v71_bucket_summary.csv", index=False)
robust_bucket_summary_df.to_csv(OUT_DIR / "v71_robust_bucket_summary.csv", index=False)
failure_month_df.to_csv(OUT_DIR / "v71_failure_monthly.csv", index=False)
failure_summary_df.to_csv(OUT_DIR / "v71_failure_summary.csv", index=False)

display_df(robust_summary_df.sort_values(["portfolio_rule", "cum_ret"], ascending=[True, False]), 40)


## 8. 暴露诊断与相对 full_v46 判断表


In [ ]:
EXPOSURE_COLS = [
    "beta", "Variance20", "ATR6", "VOL10", "DAVOL10", "VOSC", "liquidity", "momentum", "Rank1M",
    "book_to_price_ratio", "earnings_yield", "cash_flow_to_price_ratio", "px_close_to_ma60", "px_drawdown_60",
]
EXPOSURE_COLS = [c for c in EXPOSURE_COLS if c in score_panel_df.columns]


def zscore_cross_section(s):
    x = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
    std = x.std()
    if pd.isnull(std) or std <= 0:
        return x * np.nan
    return (x - x.mean()) / std


exposure_rows = []
if len(score_panel_df) and len(EXPOSURE_COLS):
    grouped = score_panel_df.groupby(["variant", "model_tag", DATE_COL])
    for (variant, model_tag, dt), gdf in progress_iter(grouped, total=grouped.ngroups, desc="selected exposure"):
        m = gdf.dropna(subset=["score", TARGET_COL]).copy()
        if m.empty:
            continue
        m[STOCK_COL] = m[STOCK_COL].astype(str)
        ordered = m.sort_values("score", ascending=False).reset_index(drop=True)
        for pspec in PORTFOLIO_SPECS:
            targets = build_board_capped_targets(ordered[STOCK_COL].astype(str).tolist(), pspec["stock_num"], pspec["board_caps"])
            sub = m[m[STOCK_COL].isin(set(targets))].copy()
            row = {"variant": variant, "model_tag": model_tag, "rebalance_date": dt, "portfolio_rule": pspec["portfolio_rule"], "stock_num": int(pspec["stock_num"])}
            for col in EXPOSURE_COLS:
                z = zscore_cross_section(m[col])
                row["z_" + col] = float(z.loc[sub.index].mean()) if len(sub) else np.nan
            exposure_rows.append(row)

exposure_monthly_df = pd.DataFrame(exposure_rows)
exposure_summary_rows = []
if len(exposure_monthly_df):
    for (variant, rule), gdf in exposure_monthly_df.groupby(["variant", "portfolio_rule"]):
        row = {"variant": variant, "portfolio_rule": rule, "months": int(len(gdf))}
        for col in exposure_monthly_df.columns:
            if col.startswith("z_"):
                row["avg_" + col] = float(pd.to_numeric(gdf[col], errors="coerce").mean())
                row["absavg_" + col] = float(pd.to_numeric(gdf[col], errors="coerce").abs().mean())
        exposure_summary_rows.append(row)
exposure_summary_df = pd.DataFrame(exposure_summary_rows)
exposure_monthly_df.to_csv(OUT_DIR / "v71_exposure_monthly.csv", index=False)
exposure_summary_df.to_csv(OUT_DIR / "v71_exposure_summary.csv", index=False)

# Decision table: compare each variant to full_v46 within robust phase for each portfolio rule.
decision_rows = []
if len(robust_summary_df):
    for rule in _bi.sorted(robust_summary_df["portfolio_rule"].dropna().unique()):
        base = robust_summary_df[(robust_summary_df["variant"] == "full_v46") & (robust_summary_df["portfolio_rule"] == rule)]
        if len(base) == 0:
            continue
        base = base.iloc[0]
        for _, row in robust_summary_df[robust_summary_df["portfolio_rule"] == rule].iterrows():
            out = row.to_dict()
            out["base_cum_ret"] = base["cum_ret"]
            out["cum_ret_ratio_vs_full"] = row["cum_ret"] / base["cum_ret"] if base["cum_ret"] not in [0, np.nan] and not pd.isnull(base["cum_ret"]) else np.nan
            out["delta_cum_ret"] = row["cum_ret"] - base["cum_ret"]
            out["delta_worst_month"] = row["worst_month"] - base["worst_month"]
            out["delta_max_drawdown"] = row["max_drawdown"] - base["max_drawdown"]
            out["delta_win_rate"] = row["monthly_win_rate"] - base["monthly_win_rate"]
            out["delta_turnover"] = row.get("avg_turnover", np.nan) - base.get("avg_turnover", np.nan)
            # Good means alpha mostly preserved and risk/failure improved. Candidate means useful defensively but not mainline.
            ratio = out["cum_ret_ratio_vs_full"]
            risk_improved = (out["delta_worst_month"] >= 0.015) or (out["delta_max_drawdown"] >= 0.020)
            if row["variant"] == "full_v46":
                verdict = "baseline"
            elif not pd.isnull(ratio) and ratio >= 0.90 and risk_improved:
                verdict = "mainline_upgrade_candidate"
            elif not pd.isnull(ratio) and ratio >= 0.75 and risk_improved:
                verdict = "defensive_candidate"
            elif not pd.isnull(ratio) and ratio < 0.70:
                verdict = "alpha_too_diluted"
            else:
                verdict = "no_clear_improvement"
            out["verdict"] = verdict
            decision_rows.append(out)

decision_df = pd.DataFrame(decision_rows)
decision_df.to_csv(OUT_DIR / "v71_decision_table_vs_full.csv", index=False)

print("saved outputs:")
for fp in _bi.sorted(OUT_DIR.glob("v71_*.csv")):
    print("-", fp)

if len(decision_df):
    display_cols = ["variant", "portfolio_rule", "stock_num", "cum_ret", "cum_ret_ratio_vs_full", "delta_worst_month", "delta_max_drawdown", "delta_win_rate", "verdict"]
    display_df(decision_df[display_cols].sort_values(["portfolio_rule", "verdict", "cum_ret_ratio_vs_full"], ascending=[True, True, False]), 60)
if len(exposure_summary_df):
    display_df(exposure_summary_df, 30)
